# Channel data

> The data to train a power control that predicts the channel: ray-traced layouts of a stable-marl environment, the agents' trajectories in them, and per link and step, what the transmitter knows and the gain it gets.

In [ ]:
#| default_exp comm.channel_data

In [ ]:
#| hide
from nbdev.showdoc import *

The transmitter of each directed link must set its power before it knows the channel. A learned predictor (see
`comm.power_control`) estimates the gain from what the transmitter does know: the link's past gains, whether the
agents moved, where the receiver was a step ago, and the layout's walls. This module produces its training data:

1. **Trajectories**: `collect_trajectories` rolls agents out in the layout of each reset seed (stable-marl's
   `MixedPolicy`) and keeps every agent's cell and heading at every step. They are kept raw, so the features can
   change without collecting again.
2. **Channels**, ray-traced (Sionna RT): only the pairs of cells the trajectories visit (`trace_visited_pairs`, into
   a `PairCache` per layout), or every pair of free cells (`trace_layouts`). Agents visit a few percent of a layout's
   pairs, so the first is far cheaper.
3. **Samples**, built from the two: `link_samples` and `build_dataset` give, per directed link and step, the features
   and the gain.

In experiments, `CachedRayTracedFading` gives comm-core the channel from a layout's `PairCache`, and traces the pairs
it has not seen yet when they first occur.

The environment comes from a config with a `_target_` (Hydra style), e.g. `configs/comm/env/findgoal.yaml`. The
layout of a stable-marl MultiGrid environment is decided by its reset seed, so a layout is named by its seed.

The config's optional `variations` lists factors of variation (stable-marl's variation space) that every reset draws
again from its seed, e.g. `[obstacles.number, clutter.number]`: FindGoal's numbers of wall blocks (0 to twice
`num_obstacles`) and of single wall cells (0 to twice `n_clutter`). A seed then still names one layout, with its own
numbers. Without it, every layout has `num_obstacles` and `n_clutter`. All of this module's envs come from `make_env`,
so the trajectories, the tracing and the samples see the same walls.

In [ ]:
#| export
from __future__ import annotations

import importlib
import json
import math
from pathlib import Path

import numpy as np

## The environment

In [ ]:
#| export
DIRS = [(1, 0), (0, 1), (-1, 0), (0, -1)]       # MultiGrid's headings: right, down, left, up

def _plain(cfg) -> dict:
    "A config as a plain dict (an OmegaConf config resolved)."
    try:
        from omegaconf import DictConfig, OmegaConf
        if isinstance(cfg, DictConfig):
            return OmegaConf.to_container(cfg, resolve=True)
    except ImportError:
        pass
    return dict(cfg)

def make_env(env_cfg):
    """A stable-marl environment from its config: a mapping with the class as `_target_` (as Hydra's) and its arguments,
    and optionally `variations`: the factors of variation every reset draws from its seed (e.g. `['obstacles.number',
    'clutter.number']`; none: the env's own values)."""
    kwargs = _plain(env_cfg)
    target = kwargs.pop('_target_')
    kwargs.pop('_convert_', None)
    variations = tuple(kwargs.pop('variations', None) or ())
    module, name = target.rsplit('.', 1)
    env = getattr(importlib.import_module(module), name)(**kwargs)
    if variations:
        known = set(_variation_names(env.unwrapped.variation_space))
        unknown = [v for v in variations if v not in known]
        if unknown:
            raise ValueError(f"unknown variations {unknown}: {sorted(known)}")
        env.unwrapped.DEFAULT_VARIATIONS = variations        # drawn again at every reset, from its seed
    return env

def _variation_names(space, prefix='') -> list[str]:
    "The dotted names of a variation space's factors (e.g. 'obstacles.number')."
    subspaces = getattr(space, 'spaces', None)
    if not isinstance(subspaces, dict):
        return [prefix[:-1]]
    return [n for k, s in subspaces.items() for n in _variation_names(s, f'{prefix}{k}.')]

In [ ]:
#| export
def free_cells(env) -> list[tuple[int, int]]:
    "The cells an agent can stand on (empty, or that can be overlapped, e.g. the goal), column by column."
    g = env.unwrapped.grid
    return [(x, y) for x in range(g.width) for y in range(g.height) if (o := g.get(x, y)) is None or o.can_overlap()]

def reset_layout(env, layout: int, edits: dict | None = None):
    """Reset `env` to `layout`: the layout of its reset seed or, for an edited layout (a key of `edits`), the layout of
    its base seed with `edits[layout]['cells']` flipped (`[x, y, 1]`: a wall added, `[x, y, 0]`: a wall removed)."""
    from stable_marl.envs.multigrid.core.world_object import Wall
    spec = (edits or {}).get(int(layout))
    env.reset(seed=int(spec['base'] if spec else layout))
    if spec:
        g = env.unwrapped.grid
        for x, y, wall in spec['cells']:
            g.set(int(x), int(y), Wall() if wall else None)
    return env

def reset_in_layout(env, layout_seed: int, rng: np.random.Generator, edits: dict | None = None):
    """Reset `env` to the layout of `layout_seed` (`reset_layout`), then move the goal and the agents to random free
    cells (random headings): many episodes in one layout. (A reset seed fixes the goal and the starts as well as the
    layout.) As FindGoal's own spawn rule: the agents on distinct cells at least `goal.min_spawn_distance` steps
    (Manhattan) from the goal, or as far as the layout allows."""
    from stable_marl.envs.multigrid.core.world_object import Goal
    reset_layout(env, layout_seed, edits)
    u = env.unwrapped
    goal = getattr(u, 'goal_pos', None)
    if goal is not None:
        u.grid.set(int(goal[0]), int(goal[1]), None)
    cells = free_cells(env)
    starts = cells
    if goal is not None:
        g = cells[rng.integers(len(cells))]
        u.goal_pos = np.asarray(g)
        u.grid.set(*g, Goal())
        far = lambda c: abs(c[0] - g[0]) + abs(c[1] - g[1])
        cells = [c for c in cells if c != g]
        try:
            wanted = int(u._variation('goal.min_spawn_distance'))
        except (AttributeError, KeyError):
            wanted = 0
        reach = min(wanted, max(map(far, cells)))
        starts = [c for c in cells if far(c) >= reach]
        if len(starts) < len(u.agents):                         # too few far cells: the farthest ones
            starts = sorted(cells, key=far)[-len(u.agents):]
    for agent, k in zip(u.agents, rng.choice(len(starts), len(u.agents), replace=False)):
        agent.state.pos = starts[k]
        agent.state.dir = int(rng.integers(4))
    return env

### Edited layouts and the distance between layouts

A layout new to the predictor can be near the training layouts or far from them. `layout_edits` makes a layout near
one: `k` of its interior cells flipped, half of them walls removed and half walls added (the number of walls stays
about the same), the free cells kept connected. An edited layout is named by an id and kept as its base seed and the
cells flipped (`reset_layout`'s `edits`); everything in this module that resets to a layout takes `edits`.

The **edit distance** between two layouts is the number of interior cells that are a wall in one and not in the
other: the fewest single-cell edits from one to the other. Between random layouts it mostly counts walls (a sparse
layout is a few edits from any other sparse one), so `layout_distance` normalizes it by the walls in either: 0 for
the same walls, 1 for no wall in common. `nearest_layout` finds the closest of a set of layouts.

In [ ]:
#| export
def interior_walls(env) -> np.ndarray:
    "The layout's interior wall cells (the outer wall left out): a boolean (width - 2, height - 2) array."
    from stable_marl.envs.multigrid.core.constants import Type
    g = env.unwrapped.grid
    return np.array([[(o := g.get(x, y)) is not None and o.type == Type.wall for y in range(1, g.height - 1)]
                     for x in range(1, g.width - 1)])

def _connected(free: set) -> bool:
    "Whether the cells are 4-connected."
    if not free:
        return False
    start = next(iter(free))
    seen, stack = {start}, [start]
    while stack:
        x, y = stack.pop()
        for c in ((x + 1, y), (x - 1, y), (x, y + 1), (x, y - 1)):
            if c in free and c not in seen:
                seen.add(c)
                stack.append(c)
    return len(seen) == len(free)

def layout_edits(env, k: int, rng: np.random.Generator, tries: int = 100) -> list[list[int]]:
    """`k` interior cells of `env`'s current layout to flip (`reset_layout`'s `cells`): `k // 2` of its walls removed
    (all if fewer) and the rest added on empty cells, with the free cells still connected."""
    from stable_marl.envs.multigrid.core.constants import Type
    g = env.unwrapped.grid
    inner = [(x, y) for x in range(1, g.width - 1) for y in range(1, g.height - 1)]
    walls = [c for c in inner if (o := g.get(*c)) is not None and o.type == Type.wall]
    empty = [c for c in inner if g.get(*c) is None]
    free = set(free_cells(env))
    n_remove = min(k // 2, len(walls))
    n_add = min(k - n_remove, len(empty) - 2)
    for _ in range(tries):
        removed = [walls[i] for i in rng.choice(len(walls), n_remove, replace=False)]
        added = [empty[i] for i in rng.choice(len(empty), n_add, replace=False)]
        if _connected((free | set(removed)) - set(added)):
            return [[int(x), int(y), 0] for x, y in removed] + [[int(x), int(y), 1] for x, y in added]
    raise RuntimeError(f"no {k} edits keep the layout connected in {tries} tries")

def edit_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    "The number of cells that are a wall in one layout and not in the other (`interior_walls`; stacks broadcast)."
    return (np.asarray(a, bool) != np.asarray(b, bool)).reshape(*np.broadcast_shapes(np.shape(a), np.shape(b))[:-2], -1).sum(-1)

def layout_distance(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    "The edit distance over the cells that are a wall in either layout: 0 the same walls, 1 none in common."
    a, b = np.asarray(a, bool), np.asarray(b, bool)
    union = (a | b).reshape(*np.broadcast_shapes(a.shape, b.shape)[:-2], -1).sum(-1)
    return np.where(union > 0, edit_distance(a, b) / np.maximum(union, 1), 0.0)

def nearest_layout(walls: np.ndarray, reference: np.ndarray) -> tuple[int, int, float]:
    "Of the `reference` layouts (a stack of `interior_walls`), the nearest to `walls`: (its index, the edit distance, the normalized distance)."
    d = layout_distance(walls[None], reference)
    i = int(np.argmin(d))
    return i, int(edit_distance(walls, reference[i])), float(d[i])

In [ ]:
cfg15 = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 15, 'num_obstacles': 6, 'n_clutter': 10}
env = make_env(cfg15)
base = interior_walls(reset_layout(env, 4))
cells = layout_edits(env, 6, np.random.default_rng(0))
assert len(cells) == 6 and sum(w for *_, w in cells) == 3                      # three walls removed, three added
edited = interior_walls(reset_layout(env, 99, {99: {'base': 4, 'cells': cells}}))
assert edit_distance(base, edited) == 6 and _connected(set(free_cells(env)))
assert abs(layout_distance(base, edited) - 6 / (base.sum() + 3)) < 1e-12 and layout_distance(base, base) == 0
others = np.stack([interior_walls(reset_layout(env, s)) for s in range(8)])
assert nearest_layout(edited, others)[:2] == (4, 6)
assert (interior_walls(reset_layout(env, 4)) == base).all()                    # the base seed itself is unchanged
rng = np.random.default_rng(1)                                                 # episodes in an edited layout keep its walls
for _ in range(3):
    reset_in_layout(env, 99, rng, {99: {'base': 4, 'cells': cells}})
    assert (interior_walls(env) == edited).all()

## Layouts

`trace_layouts` ray-traces the layout of each seed into `<out_dir>/layout_<seed>.npz`: `H[rx, tx]` (complex, summed
over the paths), the free `cells` in H's order, the `walls` grid (`walls[x, y]` = 1 where an agent cannot stand), and
what produced them (`source`). It needs Sionna RT and comm-core. Tracing is the expensive step: its time grows with
the square of the number of free cells (a 9 × 9 FindGoal grid, 39 free cells: 2–5 minutes on a CPU; 15 × 15 with 6
obstacles, about 140: roughly 13 times longer), much faster on a GPU. Layouts already traced are kept, so a long
run can be split over jobs (`seeds[i::n]`).

In [ ]:
#| export
def _scene_config(scene: dict | None):
    "stable-marl's `SceneConfig`, with `scene`'s fields replaced (dict fields, e.g. `materials`, updated)."
    from stable_marl.envs.multigrid.sionna_export import SceneConfig
    config = SceneConfig()
    for key, value in (scene or {}).items():
        if isinstance(getattr(config, key), dict):
            getattr(config, key).update(value)
        else:
            setattr(config, key, value)
    return config

def trace_layouts(
    env_cfg, # The environment's config (`make_env`)
    seeds, # Reset seeds: one layout each
    out_dir, # Where `layout_<seed>.npz` go
    frequency: float = 2.4e9, # Carrier frequency (Hz)
    max_depth: int = 5, # Reflections per path
    scene: dict | None = None, # `SceneConfig` fields to change, e.g. {'materials': {'inner_wall': 'brick'}}
    overwrite: bool = False, # Trace again the layouts already in `out_dir`
    progress: bool = True, # Print each layout's tracing time
    edits: dict | None = None, # The edited layouts among `seeds` (`reset_layout`)
    **solver_kwargs, # Other options of Sionna RT's PathSolver, e.g. samples_per_src
) -> list[Path]:
    "Ray-trace the layout of each reset seed (or edited layout) into `<out_dir>/layout_<seed>.npz`."
    import time
    from stable_marl.envs.multigrid.sionna_export import trace_channel_table
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for seed in seeds:
        path = out_dir / f'layout_{seed}.npz'
        paths.append(path)
        if path.exists() and not overwrite:
            continue
        env = reset_layout(make_env(env_cfg), seed, edits)
        start = time.time()
        table = trace_channel_table(env, _scene_config(scene), frequency, max_depth, **solver_kwargs)
        source = dict(env=_plain(env_cfg), seed=int(seed), frequency=frequency, max_depth=max_depth, scene=scene or {},
                      solver=solver_kwargs, edit=(edits or {}).get(int(seed)))
        np.savez_compressed(path, H=table['H'].astype(np.complex64), cells=table['cells'].astype(np.int16),
                            walls=table['walls'].astype(np.uint8), source=json.dumps(source, default=str))
        if progress:
            print(f'layout {seed}: {len(table["cells"])} cells, traced in {time.time() - start:.0f} s', flush=True)
    return paths

In [ ]:
#| export
def wall_geometry(walls: np.ndarray, a, b) -> tuple[float, int]:
    "Distance (cells) between cells `a` and `b`, and the number of wall cells the straight line between their centres crosses."
    n = max(int(abs(a[0] - b[0]) + abs(a[1] - b[1])) * 4, 1)
    xs, ys = np.linspace(a[0], b[0], n + 1), np.linspace(a[1], b[1], n + 1)
    crossed = {(int(round(x)), int(round(y))) for x, y in zip(xs, ys)} - {tuple(a), tuple(b)}
    return math.hypot(a[0] - b[0], a[1] - b[1]), int(sum(walls[c] for c in crossed))

def _pair(a, b) -> tuple:
    "An unordered pair of cells: a sorted tuple of two (x, y) tuples."
    a, b = tuple(int(v) for v in a), tuple(int(v) for v in b)
    return (a, b) if a <= b else (b, a)

CHANNELS = ('coherent', 'large_scale')

class ChannelTable:
    """A traced layout: the gain between two of its free cells, and its walls. From a full table, or a `PairCache`'s
    pairs. The gain is the `coherent` one, |Σ aᵢ|² (the paths' amplitudes added at the cell's centre: one small-scale
    draw), or the `large_scale` one, Σ |aᵢ|² (their powers: path loss and shadowing, a pair cache's since the powers are
    kept), with `direct_fraction` the direct path's share of it (a Rician fading's K / (K + 1))."""

    def __init__(
        self,
        path, # A `layout_<seed>.npz` of `trace_layouts`, or a `pairs_<seed>.npz` of a `PairCache`
        symmetric: bool = True, # Full tables: average the two directions of each pair (the channel is reciprocal; the tracing samples each direction apart)
    ):
        z = np.load(path)
        self.cells = [tuple(int(v) for v in c) for c in z['cells']]
        self.idx = {c: i for i, c in enumerate(self.cells)}
        self.walls = z['walls']
        self.source = json.loads(str(z['source'])) if 'source' in z.files else {}
        self.power = self.direct = None
        if 'pairs' in z.files:                                     # a pair cache: each pair traced one way
            self.G = None
            keys = [_pair(a, b) for a, b in z['pairs'].tolist()]
            self.pairs = {k: max(abs(complex(h)) ** 2, 1e-15) for k, h in zip(keys, z['H'])}
            if 'P' in z.files:                                     # the paths' powers, and the direct path's
                self.power = {k: max(float(p), 1e-15) for k, p in zip(keys, z['P'])}
                self.direct = {k: float(d) for k, d in zip(keys, z['D'])}
        else:
            G = np.abs(z['H'].astype(np.complex128)) ** 2
            self.G = np.maximum((G + G.T) / 2 if symmetric else G, 1e-15)        # G[rx, tx]; no path: -150 dB
            self.pairs = None

    def gain(self, tx, rx, channel: str = 'coherent') -> float:
        "|h|² from cell `tx` to cell `rx`, `channel` one of `CHANNELS` (from a pair cache: KeyError if the pair was not traced)."
        if channel == 'large_scale':
            if self.power is None:
                raise ValueError("this layout was traced without the paths' powers (a full table, or a pair cache from before "
                                 "they were kept): trace it again (into a new layout directory) for the large-scale channel")
            return self.power[_pair(tx, rx)]
        if channel != 'coherent':
            raise ValueError(f'channel: one of {CHANNELS}, not {channel!r}')
        if self.pairs is not None:
            return self.pairs[_pair(tx, rx)]
        return float(self.G[self.idx[tuple(rx)], self.idx[tuple(tx)]])

    def direct_fraction(self, tx, rx) -> float:
        "The direct path's share of the large-scale power between two cells (0: none, or unknown)."
        if self.power is None:
            return 0.0
        k = _pair(tx, rx)
        return float(min(self.direct[k] / self.power[k], 1.0))

def table_path(layout_dir, seed) -> Path:
    "The layout's full table (`layout_<seed>.npz`) if there is one, else its pair cache (`pairs_<seed>.npz`)."
    full = Path(layout_dir) / f'layout_{seed}.npz'
    return full if full.exists() else Path(layout_dir) / f'pairs_{seed}.npz'

## Trajectories

`collect_trajectories` rolls out `episodes` episodes in the layout of each seed, each from random starts and goal
(`reset_in_layout`), and saves `trajectories_<seed>.npz`: `cells` (episodes, steps, agents, 2) and `dirs`
(episodes, steps, agents), with -1 once an agent has left the grid (FindGoal removes an agent at the goal) or the
episode has ended, and the `behaviour` of each episode. The agents follow stable-marl's `MixedPolicy`, which draws a
behaviour per episode with the probabilities in `policy`:

* `random`: uniform actions.
* `epsilon_expert`: a shortest path to the goal, with a random action with probability `epsilon`.
* `waypoint`: a shortest path to a random cell of the agent's own, then to the goal.

The movement in the data should be the movement at deployment: the predictor learns how the gain changes when the
agents move.

In [ ]:
#| export
def collect_trajectories(
    env_cfg, # The environment's config (`make_env`)
    seeds, # The layouts' reset seeds
    out_dir, # Where `trajectories_<seed>.npz` go
    episodes: int = 5, # Episodes per layout
    max_steps: int = 150, # Steps per episode, at most
    policy: dict | None = None, # `MixedPolicy`'s behaviour -> probability; None: 0.2 random, 0.5 epsilon_expert, 0.3 waypoint
    epsilon: float = 0.3, # The random actions of `epsilon_expert`
    seed: int = 0, # Seed of the starts and the actions
    overwrite: bool = False, # Collect again the layouts already in `out_dir` (one with fewer episodes is, always)
    edits: dict | None = None, # The edited layouts among `seeds` (`reset_layout`)
) -> list[Path]:
    """Every agent's cell and heading at every step of `episodes` episodes in the layout of each seed. A layout's
    episodes come from its own seeded generator, one after the other: collected again with more episodes, its first
    ones are the same."""
    from stable_marl.envs.multigrid.expert import MixedPolicy
    weights = _plain(policy) if policy is not None else None
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for layout in seeds:
        path = out_dir / f'trajectories_{layout}.npz'
        paths.append(path)
        if path.exists() and not overwrite and len(np.load(path)['cells']) >= episodes:
            continue
        rng = np.random.default_rng([seed, int(layout)])
        agents_policy = MixedPolicy(weights, epsilon, seed=int(rng.integers(2 ** 63)))
        env = make_env(env_cfg)
        n = len(env.unwrapped.agents)
        cells = np.full((episodes, max_steps, n, 2), -1, np.int16)
        dirs = np.full((episodes, max_steps, n), -1, np.int8)
        used = []
        for e in range(episodes):
            reset_in_layout(env, layout, rng, edits)
            u = env.unwrapped
            for t in range(max_steps):
                for i, agent in enumerate(u.agents):
                    if not agent.state.terminated and tuple(agent.state.pos) != (-1, -1):
                        cells[e, t, i] = agent.state.pos
                        dirs[e, t, i] = int(agent.state.dir)
                actions = agents_policy.act(u)
                if t == 0:
                    used.append(agents_policy.behaviour(u))
                _, _, terminated, truncated, _ = env.step({i: int(a) for i, a in enumerate(actions)})
                if all(terminated.values()) or all(truncated.values()):
                    break
        np.savez_compressed(path, cells=cells, dirs=dirs, behaviour=np.array(used), layout=int(layout))
    return paths

## Tracing the visited pairs

Agents visit a few percent of a layout's pairs of cells, and a Sionna RT call costs about the same per pair, so
tracing only those is far cheaper than a full table (on a 15 × 15 FindGoal grid: a few minutes per layout instead of
about 45, on a CPU). A `PairCache` keeps a layout's traced pairs in `pairs_<seed>.npz`, and traces the missing ones:

* each pair once, one way (the channel is reciprocal; this also avoids the two directions' sampling mismatch);
* grouped by transmitter, the cell of each pair shared by more pairs transmitting, so that one Sionna RT call traces
  several pairs.

`trace_visited_pairs` traces the pairs of each layout's trajectories. The cache grows as needed: more episodes, or the
pairs met in an experiment (`CachedRayTracedFading`), are traced when first needed.

In [ ]:
#| export
DIRECT_TOLERANCE = 0.2e-9                                         # s: a path this close to the straight line's delay is the direct one

class PairCache:
    """A layout's traced pairs of cells, kept in `path`; `trace` traces the missing ones (one way each, grouped by
    transmitter). Per pair: the coherent channel `H` (the paths' amplitudes added), the large-scale power `P` (their
    powers added) and the direct path's power `D` (the path with the straight line's delay, within `DIRECT_TOLERANCE`:
    line of sight, or straight through thin walls). A cache traced with other settings (frequency, depth, scene,
    solver) is refused."""

    def __init__(
        self,
        path, # The cache file: `<dir>/pairs_<seed>.npz` (read if it exists)
        env_cfg, # The environment's config (`make_env`)
        layout_seed: int, # The layout's reset seed
        frequency: float = 2.4e9, # Carrier frequency (Hz)
        max_depth: int = 5, # Reflections per path
        scene: dict | None = None, # `SceneConfig` fields to change
        tracer=None, # tracer(tx_cell, rx_cells) -> per receiver its complex channel, or (channel, power, direct power); None: Sionna RT
        edits: dict | None = None, # The edited layouts (`reset_layout`), if `layout_seed` is one
        **solver_kwargs, # Other options of comm-core's `trace_paths` (e.g. batch) and Sionna RT's PathSolver
    ):
        self.path, self.env_cfg, self.seed = Path(path), env_cfg, int(layout_seed)
        self.frequency, self.max_depth, self.scene, self.solver_kwargs = frequency, max_depth, scene or {}, solver_kwargs
        self.tracer, self._scene, self.edit = tracer, None, (edits or {}).get(self.seed)
        self.env = reset_layout(make_env(env_cfg), self.seed, edits)
        self.cells = free_cells(self.env)
        g = self.env.unwrapped.grid
        self.walls = np.ones((g.width, g.height), np.uint8)
        for c in self.cells:
            self.walls[c] = 0
        self.H, self.P, self.D, self.unsaved, self.powers = {}, {}, {}, 0, True
        if self.path.exists():
            z = np.load(self.path)
            if [tuple(c) for c in z['cells'].tolist()] != self.cells:
                raise ValueError(f"{self.path} holds another layout than layout {self.seed}'s")
            if 'source' in z.files:
                was = json.loads(str(z['source']))
                then = json.dumps({k: was.get(k) for k in ('frequency', 'max_depth', 'scene', 'solver')}, sort_keys=True, default=str)
                if then != self._settings():
                    raise ValueError(f"{self.path} was traced with other settings ({then}) than these ({self._settings()}): "
                                     "trace into another layout directory")
            keys = [_pair(a, b) for a, b in z['pairs'].tolist()]
            self.H = {k: complex(h) for k, h in zip(keys, z['H'])}
            self.powers = 'P' in z.files                            # a cache from before the powers were kept: none
            if self.powers:
                self.P = {k: float(p) for k, p in zip(keys, z['P'])}
                self.D = {k: float(d) for k, d in zip(keys, z['D'])}

    def _settings(self) -> str:
        return json.dumps(dict(frequency=self.frequency, max_depth=self.max_depth, scene=self.scene, solver=self.solver_kwargs),
                          sort_keys=True, default=str)

    def missing(self, pairs) -> list[tuple]:
        "The pairs (of distinct cells) not traced yet."
        return sorted({p for p in (_pair(a, b) for a, b in pairs) if p[0] != p[1] and p not in self.H})

    def trace(self, pairs, progress: bool = False) -> int:
        "Trace the missing `pairs`; the number traced."
        from collections import Counter, defaultdict
        todo = self.missing(pairs)
        if todo and not self.powers:
            raise ValueError(f"{self.path} was traced before the paths' powers were kept: trace into another layout directory")
        degree = Counter(c for p in todo for c in p)
        groups = defaultdict(list)
        for a, b in todo:
            tx, rx = (a, b) if (degree[a], a) >= (degree[b], b) else (b, a)
            groups[tx].append(rx)
        for k, (tx, rxs) in enumerate(groups.items()):
            for rx, r in zip(rxs, self._trace(tx, rxs)):
                h, p, d = r if isinstance(r, tuple) else (r, abs(complex(r)) ** 2, 0.0)   # a stand-in's channel: no direct path
                k = _pair(tx, rx)
                self.H[k], self.P[k], self.D[k] = complex(h), float(p), float(d)
            if progress:
                print(f'{k + 1} / {len(groups)} transmitters', end='\r')
        self.unsaved += len(todo)
        return len(todo)

    def _trace(self, tx, rxs):
        if self.tracer is not None:
            return self.tracer(tx, rxs)
        import contextlib, io, os, tempfile
        import sionna.rt as rt
        from comm_core import trace_paths
        from stable_marl.envs.multigrid.sionna_export import export_scene, grid_to_world
        config = _scene_config(self.scene)
        if self._scene is None:                                   # the layout's scene, built once
            self._dir = tempfile.TemporaryDirectory()
            with contextlib.redirect_stdout(io.StringIO()):
                export_scene(self.env, self._dir.name, config, frequency=self.frequency)
            self._scene = rt.load_scene(os.path.join(self._dir.name, 'scene.xml'))
            self._scene.frequency = self.frequency
        g = self.env.unwrapped.grid
        point = lambda c: grid_to_world(c[0], c[1], g.width, g.height, config, config.antenna_height)
        paths = trace_paths(self._scene, [point(tx)], [point(c) for c in rxs], max_depth=self.max_depth, **self.solver_kwargs)
        a, tau = paths['a'][0], paths['tau'][0]                   # (receivers, paths)
        straight = np.linalg.norm(paths['rx_points'] - paths['tx_points'][0], axis=1) / 299792458.0
        direct = (np.abs(a) > 0) & (np.abs(tau - straight[:, None]) < DIRECT_TOLERANCE)
        power = (np.abs(a) ** 2).sum(-1)
        return [(complex(h), float(p), float(d)) for h, p, d in zip(a.sum(-1), power, (np.abs(a) ** 2 * direct).sum(-1))]

    def gain(self, a, b, channel: str = 'coherent') -> float:
        "|h|² between cells `a` and `b` (`ChannelTable.gain`'s `channel`), traced now if it was not."
        self.trace([(a, b)])
        k = _pair(a, b)
        return max(self.P[k] if channel == 'large_scale' else abs(self.H[k]) ** 2, 1e-15)

    def save(self):
        "Write the cache to `path`."
        pairs = sorted(self.H)
        source = dict(env=_plain(self.env_cfg), seed=self.seed, frequency=self.frequency, max_depth=self.max_depth,
                      scene=self.scene, solver=self.solver_kwargs, edit=self.edit)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        powers = dict(P=np.array([self.P[p] for p in pairs], np.float64), D=np.array([self.D[p] for p in pairs], np.float64)) \
            if self.powers else {}
        np.savez_compressed(self.path, pairs=np.array(pairs, np.int16).reshape(-1, 2, 2),
                            H=np.array([self.H[p] for p in pairs], np.complex64), cells=np.array(self.cells, np.int16),
                            walls=self.walls, source=json.dumps(source, default=str), **powers)
        self.unsaved = 0

In [ ]:
#| export
def visited_pairs(trajectories) -> set[tuple]:
    "The pairs of cells two agents were in at the same step, in a `trajectories_<seed>.npz` (a path, or the loaded arrays)."
    cells = (np.load(trajectories) if isinstance(trajectories, (str, Path)) else trajectories)['cells']
    on = cells[..., 0] >= 0
    out = set()
    for i in range(cells.shape[2]):
        for j in range(i + 1, cells.shape[2]):
            both = on[..., i] & on[..., j]
            for a, b in zip(cells[..., i, :][both].tolist(), cells[..., j, :][both].tolist()):
                if a != b:
                    out.add(_pair(a, b))
    return out

def trace_visited_pairs(
    env_cfg, # The environment's config (`make_env`)
    trajectory_dir, # The `trajectories_<seed>.npz` of `collect_trajectories`
    seeds, # The layouts' reset seeds
    out_dir, # Where the `pairs_<seed>.npz` caches are (and go)
    frequency: float = 2.4e9, # Carrier frequency (Hz)
    max_depth: int = 5, # Reflections per path
    scene: dict | None = None, # `SceneConfig` fields to change
    progress: bool = True, # Print each layout's count and time
    tracer=None, # A stand-in for Sionna RT (see `PairCache`)
    edits: dict | None = None, # The edited layouts among `seeds` (`reset_layout`)
    **solver_kwargs, # Other options of `trace_paths` and Sionna RT's PathSolver
) -> list[Path]:
    "Trace the pairs of cells each layout's trajectories visit, into its `PairCache` (only the pairs not traced yet)."
    import time
    paths = []
    for seed in seeds:
        cache = PairCache(Path(out_dir) / f'pairs_{seed}.npz', env_cfg, seed, frequency, max_depth, scene, tracer, edits,
                          **solver_kwargs)
        start = time.time()
        pairs = visited_pairs(Path(trajectory_dir) / f'trajectories_{seed}.npz')
        traced = cache.trace(pairs)
        cache.save()
        paths.append(cache.path)
        if progress:
            print(f'layout {seed}: {len(pairs)} pairs visited, {traced} traced now, in {time.time() - start:.0f} s', flush=True)
    return paths

In an experiment, `CachedRayTracedFading` is the channel comm-core uses (e.g. `LinkLevelChannel(fading=...)`): the
application places the nodes on cells (`set_positions`, as for comm-core's `RayTracedFading`), and a pair not in the
cache is traced the first time it is needed (about a second on a CPU), then kept. The cache is written every
`save_every` new pairs, and by `save`.

In [ ]:
#| export
from comm_core import Fading

class CachedRayTracedFading(Fading):
    "The channel between the nodes' cells, from a layout's `PairCache`; a pair not traced yet is traced when first needed."

    def __init__(
        self,
        cache: PairCache, # The layout's cache
        save_every: int = 50, # Write the cache after this many new pairs
        channel: str = 'coherent', # `ChannelTable`'s: coherent, or large_scale (its amplitude, √P)
    ):
        self.cache, self.save_every, self.node_positions, self.channel = cache, save_every, {}, channel

    def set_positions(self, positions):
        "Where the nodes are now: {node: (x, y)}."
        self.node_positions.update({k: tuple(int(v) for v in p) for k, p in positions.items()})

    def gain_between(self, tx_position, rx_position) -> float:
        "|h|² between two cells (inf: the same cell)."
        a, b = tuple(int(v) for v in tx_position), tuple(int(v) for v in rx_position)
        return math.inf if a == b else self.cache.gain(a, b, self.channel)

    def coefficient(self, transmission, link) -> complex:
        a, b = self.node_positions[transmission.sender], self.node_positions[transmission.receiver]
        if a == b:
            return math.inf
        if self.cache.trace([(a, b)]) and self.cache.unsaved >= self.save_every:
            self.cache.save()
        k = _pair(a, b)
        h = self.cache.H[k] if self.channel == 'coherent' else complex(math.sqrt(self.cache.P[k]))
        return h if abs(h) ** 2 >= 1e-15 else complex(math.sqrt(1e-15))   # no path: -150 dB

    def save(self):
        "Write the cache."
        self.cache.save()

## Samples

For each directed link `j → i` and step `t ≥ history` at which both agents are on the grid (and were, `history`
steps back), the transmitter knows:

* the link's last `history` gains, fed back by the receiver (most recent first);
* whether it moved this step, and whether the receiver moved a step ago (the receiver's report is a step late);
* the receiver's cell and heading a step ago: since then, it has stayed or stepped ahead, its *possible cells*;
* the layout's walls.

`features` turns these into the predictor's inputs, for one cell the receiver may be in: the log-distance and the
distance to it, the walls in between, the past gains in dB, and the two moves. For training, it is computed for
the receiver's actual cell (`X`); for deployment, for each of its possible cells (`X_possible`, with
`possible_mask`), where the predictor takes the lowest prediction.

The gain is the table's `channel`: the `coherent` one (the paths' amplitudes added at the cell's centre, which holds
one small-scale draw, frozen in space), or the `large_scale` one (their powers added: path loss and shadowing). On the
large-scale channel, `fading='rician'` adds the small-scale fading the ray tracer cannot give a cell (an agent is
anywhere in it, and the scene is static): per pair of agents (the same both ways), Rician with the direct path's share
of the power from the table (`direct_fraction`; none: Rayleigh), drawn again whenever an agent moves (a cell is many
wavelengths) and drifting while both stand still, `ρ = J₀(2π f_D Δt)` (`fading_correlation`: the step's duration and
the environment's Doppler).

In [ ]:
#| export
def db(g):
    "Gains in dB."
    return 10 * np.log10(np.maximum(np.asarray(g, float), 1e-30))

def features(walls: np.ndarray, tx, rx, past_gains, tx_moved: bool, rx_moved: bool) -> list[float]:
    "The predictor's inputs for a transmitter at `tx` and a receiver at `rx`: geometry, past gains (dB, most recent first), moves."
    distance, crossed = wall_geometry(walls, tx, rx)
    return [math.log10(max(distance, 0.5)), distance, crossed, *db(past_gains), float(tx_moved), float(rx_moved)]

STEP_FEATURES = 6                                                   # `step_features`' length

def step_features(walls: np.ndarray, tx, rx, gain: float, tx_moved: bool, rx_moved: bool) -> list[float]:
    "One past step of a link, for the sequence predictors: its gain (dB), the geometry between the two agents then, and whether each had moved."
    distance, crossed = wall_geometry(walls, tx, rx)
    return [float(db(gain)), math.log10(max(distance, 0.5)), distance, crossed, float(tx_moved), float(rx_moved)]

In [ ]:
#| export
FADINGS = (None, 'rician')

def fading_correlation(step_seconds: float = 1.0, doppler_hz: float = 0.1) -> float:
    """ρ of the small-scale fading between two steps while both agents stand still: Jakes' J₀(2π f_D Δt), the
    environment's Doppler `doppler_hz` (people walking by, doors) over a step of `step_seconds`, in [0, 1)."""
    from scipy.special import j0
    return float(np.clip(j0(2 * np.pi * doppler_hz * step_seconds), 0.0, 0.9999))

def _fading(cells: np.ndarray, rng: np.random.Generator, rho: float, direct=None) -> np.ndarray:
    """|g_t|² of one pair of agents' small-scale fading over an episode's steps (cells: (T, 2, 2)), of mean 1: Rician,
    `direct[t]` the direct path's share of the power (K / (K + 1); 0: Rayleigh). When an agent moves (a cell is many
    wavelengths) the scattered part and the direct path's phase are drawn again; while both stand still the scattered
    part drifts, `ρ` from one step to the next, and the direct path stays."""
    direct = np.zeros(len(cells)) if direct is None else np.clip(np.asarray(direct, float), 0.0, 1.0)
    draw = lambda: (rng.standard_normal() + 1j * rng.standard_normal()) / np.sqrt(2)
    w, phase, out = draw(), rng.uniform(0, 2 * np.pi), np.ones(len(cells))
    for t in range(len(cells)):
        if t and not np.array_equal(cells[t], cells[t - 1]):
            w, phase = draw(), rng.uniform(0, 2 * np.pi)
        elif t:
            w = rho * w + np.sqrt(1 - rho ** 2) * draw()
        out[t] = abs(np.sqrt(direct[t]) * np.exp(1j * phase) + np.sqrt(1 - direct[t]) * w) ** 2
    return out

def episode_fading(cells: np.ndarray, rng: np.random.Generator, rho: float = 0.9, table: ChannelTable | None = None) -> dict:
    """Each pair of agents' small-scale fading |g_t|² over an episode (cells: (steps, agents, 2)): {(i, j): (steps,)},
    i < j, the same both ways; Rician with the direct path's share from `table` (None: Rayleigh)."""
    n = cells.shape[1]
    on = cells[..., 0] >= 0
    def direct(i, j):
        if table is None:
            return None
        return [table.direct_fraction(cells[t, i], cells[t, j]) if on[t, i] and on[t, j] and tuple(cells[t, i]) != tuple(cells[t, j])
                else 0.0 for t in range(len(cells))]
    return {(i, j): _fading(cells[:, [i, j]], np.random.default_rng(rng.integers(2 ** 63)), rho, direct(i, j))
            for i in range(n) for j in range(i + 1, n)}

def link_samples(
    table: ChannelTable, # The layout's channel
    cells: np.ndarray, # One episode's cells: (steps, agents, 2), -1 when an agent is not on the grid
    dirs: np.ndarray, # Its headings: (steps, agents)
    history: int = 4, # Past gains the transmitter knows
    fading: str | None = None, # None, or 'rician': small-scale fading on top of the large-scale channel (`FADINGS`)
    fading_correlation: float = 0.9, # ρ of the fading between steps while both agents stand still (`fading_correlation`)
    rng: np.random.Generator | None = None, # The fading's randomness
    steps: bool = False, # Also `S`: each past step's `step_features` (most recent first), for the per-step sequence predictors
    channel: str = 'coherent', # The table's (`ChannelTable.gain`): coherent, or large_scale (with the fading: rician)
) -> dict:
    """Per directed link and step: the features (to the receiver's actual cell, and to each of its possible cells) and the
    gain; with `steps`, also the past steps' gains, geometry and moves (`S`: samples × history × `STEP_FEATURES`)."""
    rng = rng if rng is not None else np.random.default_rng(0)
    if fading not in FADINGS:
        raise ValueError(f"fading: one of {FADINGS}, not {fading!r} (Rayleigh: 'rician' on a table without a direct path)")
    if fading is not None and channel != 'large_scale':
        raise ValueError("the coherent channel already holds a small-scale draw: fading on top of the large_scale channel only")
    T, n = cells.shape[:2]
    fades = episode_fading(cells, rng, fading_correlation, table) if fading == 'rician' else None
    on = (cells[..., 0] >= 0)
    X, Xp, mask, gain, link, S = [], [], [], [], [], []
    for j in range(n):
        for i in range(n):
            if i == j:
                continue
            both = on[:, i] & on[:, j]
            g = np.zeros(T)
            for t in np.flatnonzero(both):
                g[t] = table.gain(cells[t, j], cells[t, i], channel)
            if fades is not None:
                g = g * fades[min(i, j), max(i, j)]
            if steps:                                               # each step's row once per link, its samples take `history` of them
                moved = lambda a, t: t > 0 and on[t - 1, a] and tuple(cells[t, a]) != tuple(cells[t - 1, a])
                step_rows = {t: step_features(table.walls, tuple(cells[t, j]), tuple(cells[t, i]), g[t], moved(j, t), moved(i, t))
                        for t in np.flatnonzero(both)}
            for t in range(max(history, 2), T):
                if not both[t - history:t + 1].all() or not on[t - 2, i]:
                    continue
                tx, rx, rx_before = tuple(cells[t, j]), tuple(cells[t, i]), tuple(cells[t - 1, i])
                past = g[t - history:t][::-1]
                moved = (tx != tuple(cells[t - 1, j]), rx_before != tuple(cells[t - 2, i]))
                d = DIRS[int(dirs[t - 1, i])]
                ahead = (rx_before[0] + d[0], rx_before[1] + d[1])
                possible = [c for c in (rx_before, ahead) if c in table.idx and c != tx] or [rx_before]
                X.append(features(table.walls, tx, rx, past, *moved))
                rows = [features(table.walls, tx, c, past, *moved) for c in possible]
                Xp.append(rows + [rows[0]] * (2 - len(rows)))
                mask.append([True] * len(rows) + [False] * (2 - len(rows)))
                gain.append(g[t])
                link.append((j, i, t))
                if steps:
                    S.append([step_rows[s] for s in range(t - 1, t - history - 1, -1)])
    F = 3 + history + 2
    out = dict(X=np.array(X, float).reshape(-1, F), X_possible=np.array(Xp, float).reshape(-1, 2, F),
               possible_mask=np.array(mask, bool).reshape(-1, 2), gain=np.array(gain, float), link=np.array(link, int).reshape(-1, 3))
    if steps:
        out['S'] = np.array(S, np.float32).reshape(-1, history, STEP_FEATURES)
    return out

In [ ]:
#| export
def build_dataset(
    layout_dir, # The layouts' channels: `layout_<seed>.npz` (`trace_layouts`) or `pairs_<seed>.npz` (`trace_visited_pairs`)
    trajectory_dir, # The `trajectories_<seed>.npz` of `collect_trajectories`
    seeds, # The layouts to use
    history: int = 4, # Past gains the transmitter knows
    fading: str | None = None, # None, or 'rician' (`link_samples`')
    fading_correlation: float = 0.9, # ρ of the fading while both agents stand still
    symmetric: bool = True, # Average the two directions of each traced pair
    seed: int = 0, # Seed of the fading
    steps: bool = False, # Also each sample's past steps (`S`, `link_samples`')
    channel: str = 'coherent', # coherent, or large_scale (`ChannelTable.gain`)
) -> dict:
    "`link_samples` of every episode in the layouts of `seeds`, concatenated, with each sample's `layout` and `episode`."
    parts = []
    for layout in seeds:
        table = ChannelTable(table_path(layout_dir, layout), symmetric=symmetric)
        z = np.load(Path(trajectory_dir) / f'trajectories_{layout}.npz')
        rng = np.random.default_rng([seed, int(layout)])
        for e, (cells, dirs) in enumerate(zip(z['cells'], z['dirs'])):
            s = link_samples(table, cells, dirs, history, fading, fading_correlation, rng, steps, channel)
            s['layout'] = np.full(len(s['gain']), int(layout))
            s['episode'] = np.full(len(s['gain']), e)
            parts.append(s)
    return {k: np.concatenate([p[k] for p in parts]) for k in parts[0]}

def with_history(data: dict, history: int) -> dict:
    """`data` (`build_dataset`'s) with only its `history` most recent past gains (and past steps, `S`), for the same
    transmissions: the data built with a longer history, several history lengths are compared on the same ones."""
    F = data['X'].shape[-1]
    built = F - 5
    if not 1 <= history <= built:
        raise ValueError(f'history: 1 to {built} (the data has {built} past gains), not {history}')
    cols = [*range(3 + history), F - 2, F - 1]
    out = {**data, 'X': data['X'][..., cols], 'X_possible': data['X_possible'][..., cols]}
    if 'S' in data:
        out['S'] = data['S'][:, :history]
    return out

## Example

A small FindGoal (9 × 9, FindGoal's 10 single-cell walls): trajectories in two layouts, then their visited pairs
traced with a stand-in for Sionna RT (free-space path loss and 6 dB per wall crossed), so that the example runs
without it. With Sionna RT, leave out `tracer`.

In [ ]:
import tempfile

In [ ]:
env_cfg = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 9, 'num_obstacles': 0, 'max_steps': 60}
tmp = Path(tempfile.mkdtemp())

def stand_in_table(seed, path):
    "Instead of tracing: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    env = make_env(env_cfg)
    env.reset(seed=seed)
    cells, g = free_cells(env), env.unwrapped.grid
    walls = np.array([[0 if (x, y) in set(cells) else 1 for y in range(g.height)] for x in range(g.width)], np.uint8)
    H = np.zeros((len(cells), len(cells)), complex)
    for r, b in enumerate(cells):
        for s, a in enumerate(cells):
            if a != b:
                d, w = wall_geometry(walls, a, b)
                H[r, s] = 0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20)
    np.savez_compressed(path, H=H, cells=np.array(cells), walls=walls, source=json.dumps({'seed': seed}))

def stand_in(walls):
    "A tracer for `PairCache`: free-space loss at 2.4 GHz (2 m cells) and 6 dB per wall crossed."
    def tracer(tx, rxs):
        return [0.125 / (4 * np.pi * 2 * d) * 10 ** (-6 * w / 20) for d, w in (wall_geometry(walls, tx, rx) for rx in rxs)]
    return tracer

collect_trajectories(env_cfg, [0, 1], tmp, episodes=3, max_steps=40, policy={'random': 0.5, 'epsilon_expert': 0.5})
for seed in (0, 1):
    walls = PairCache(tmp / 'unused.npz', env_cfg, seed).walls
    trace_visited_pairs(env_cfg, tmp, [seed], tmp, tracer=stand_in(walls))
data = build_dataset(tmp, tmp, [0, 1], channel='large_scale', fading='rician')
print({k: v.shape for k, v in data.items()})

In [ ]:
table, z = ChannelTable(tmp / 'pairs_0.npz'), np.load(tmp / 'trajectories_0.npz')
assert z['cells'].shape == (3, 40, 2, 2) and set(z['behaviour']) <= {'random', 'epsilon_expert'}
on = z['cells'][..., 0] >= 0
assert all(tuple(c) in table.idx for c in z['cells'][on])                      # agents only on free cells
assert set(table.pairs) == visited_pairs(z)                                     # exactly the visited pairs
assert len({tuple(map(tuple, z['cells'][e, 0])) for e in range(3)}) == 3        # different starts in one layout
assert data['X'].shape[1] == 3 + 4 + 2 and (data['possible_mask'][:, 0]).all()
assert set(data['episode']) == {0, 1, 2} and (np.diff(data['episode'][data['layout'] == 0]) >= 0).all()
more = Path(tempfile.mkdtemp())                                                 # more episodes: the first ones the same
collect_trajectories(env_cfg, [0], more, episodes=2, max_steps=40, policy={'random': 0.5, 'epsilon_expert': 0.5})
collect_trajectories(env_cfg, [0], more, episodes=4, max_steps=40, policy={'random': 0.5, 'epsilon_expert': 0.5})
z4 = np.load(more / 'trajectories_0.npz')
assert len(z4['cells']) == 4 and (z4['cells'][:3] == z['cells']).all() and (z4['behaviour'][:3] == z['behaviour']).all()
s = link_samples(table, z['cells'][0], z['dirs'][0])                            # no fading: the table's gains
assert np.allclose(s['gain'], [table.gain(z['cells'][0, t, j], z['cells'][0, t, i]) for j, i, t in s['link']])

In [ ]:
# the past steps (S): each step's gain is the features' past gain, its geometry the agents' cells then, its moves since the step before
s = link_samples(table, z['cells'][0], z['dirs'][0], history=6, channel='large_scale', fading='rician', rng=np.random.default_rng(1), steps=True)
assert s['S'].shape == (len(s['gain']), 6, STEP_FEATURES) and np.allclose(s['S'][:, :, 0], s['X'][:, 3:9], atol=1e-4)
cells = z['cells'][0]
for (j, i, t), row in zip(s['link'][::7], s['S'][::7]):
    for k, f in enumerate(row):                                                # k = 0: the step before t
        u = t - 1 - k
        d, w = wall_geometry(table.walls, tuple(cells[u, j]), tuple(cells[u, i]))
        assert np.allclose(f[1:4], [math.log10(max(d, 0.5)), d, w], atol=1e-4)
        assert f[4] == float(u > 0 and tuple(cells[u, j]) != tuple(cells[u - 1, j]))
        assert f[5] == float(u > 0 and tuple(cells[u, i]) != tuple(cells[u - 1, i]))
# with_history: the most recent past gains and steps, the same transmissions
d6 = build_dataset(tmp, tmp, [0, 1], history=6, steps=True)
d3 = with_history(d6, 3)
assert d3['X'].shape[1] == 3 + 3 + 2 and d3['S'].shape[1] == 3 and len(d3['gain']) == len(d6['gain'])
assert np.array_equal(d3['X'][:, :6], d6['X'][:, :6]) and np.array_equal(d3['X'][:, -2:], d6['X'][:, -2:])
assert np.array_equal(d3['X_possible'][..., -2:], d6['X_possible'][..., -2:]) and np.array_equal(d3['S'], d6['S'][:, :3])
assert 'S' not in data and with_history(data, 4)['X'].shape == data['X'].shape
try:
    with_history(d6, 7)
    raise AssertionError('no error')
except ValueError:
    pass

In [ ]:
# the large-scale channel: the paths' powers and the direct path's kept per pair; Rician fading on top of it
three = Path(tempfile.mkdtemp())
walls0 = PairCache(three / 'unused.npz', env_cfg, 0).walls
def with_direct(tx, rxs):                                                       # a tracer returning (channel, power, direct power)
    return [(complex(h), 2 * abs(h) ** 2, abs(h) ** 2 if w == 0 else 0.0)        # line of sight: half the power direct
            for h, (_, w) in zip(stand_in(walls0)(tx, rxs), (wall_geometry(walls0, tx, rx) for rx in rxs))]
cache = PairCache(three / 'pairs_0.npz', env_cfg, 0, tracer=with_direct, max_depth=8)
pairs = sorted(visited_pairs(tmp / 'trajectories_0.npz'))[:40]
cache.trace(pairs)
cache.save()
t3 = ChannelTable(three / 'pairs_0.npz')
for a, b in pairs:
    w = wall_geometry(walls0, a, b)[1]
    assert np.isclose(t3.gain(a, b, 'large_scale'), 2 * t3.gain(a, b)) and t3.direct_fraction(a, b) == (0.5 if w == 0 else 0.0)
stand_in_table(0, full := three / 'layout_0.npz')
assert ChannelTable(tmp / 'pairs_0.npz').direct_fraction(*pairs[0]) == 0.0         # a stand-in's: no direct path (Rayleigh)
for bad in (lambda: PairCache(three / 'pairs_0.npz', env_cfg, 0, max_depth=5),     # traced with other settings: refused
            lambda: ChannelTable(full).gain(*pairs[0], 'large_scale'),                  # a full table: no powers
            lambda: link_samples(table, z['cells'][0], z['dirs'][0], fading='rician'),          # fading on the coherent channel
            lambda: link_samples(table, z['cells'][0], z['dirs'][0], fading='rayleigh', channel='large_scale')):
    try:
        bad()
        raise AssertionError('no error')
    except ValueError:
        pass
# the small-scale fading: a direct path alone does not fade; Rayleigh has mean 1; it drifts while both stand still
still = np.zeros((4000, 2, 2), int)
assert np.allclose(_fading(still, np.random.default_rng(0), 0.9, np.ones(4000)), 1.0)
f = _fading(still, np.random.default_rng(0), 0.0)
assert abs(f.mean() - 1) < 0.08 and abs(np.corrcoef(f[:-1], f[1:])[0, 1]) < 0.06            # ρ = 0: independent steps
g = _fading(still, np.random.default_rng(0), 0.99)
assert np.corrcoef(g[:-1], g[1:])[0, 1] > 0.9
moving = np.stack([np.stack([np.arange(4000) % 2, np.zeros(4000, int)], -1), np.zeros((4000, 2), int)], 1)
m = _fading(moving, np.random.default_rng(0), 0.99)                             # every step a move: drawn again each time
assert abs(np.corrcoef(m[:-1], m[1:])[0, 1]) < 0.06
assert fading_correlation(1.0, 0.0) == 0.9999 and abs(fading_correlation(1.0, 0.1) - 0.9037) < 1e-3   # J₀(0.2π)
s = link_samples(t3, z['cells'][0], z['dirs'][0], channel='large_scale', fading='rician', rng=np.random.default_rng(0))
s0 = link_samples(t3, z['cells'][0], z['dirs'][0], channel='large_scale')
assert np.array_equal(s['link'], s0['link']) and not np.allclose(s['gain'], s0['gain'])

In [ ]:
# an edited layout: collected and traced as any other, through `edits`
edits = {100: {'base': 0, 'cells': layout_edits(reset_layout(make_env(env_cfg), 0), 4, np.random.default_rng(0))}}
collect_trajectories(env_cfg, [100], tmp, episodes=2, max_steps=40, policy={'random': 1.0}, edits=edits)
walls = PairCache(tmp / 'unused.npz', env_cfg, 100, edits=edits).walls
trace_visited_pairs(env_cfg, tmp, [100], tmp, tracer=stand_in(walls), edits=edits)
edited, plain = ChannelTable(tmp / 'pairs_100.npz'), ChannelTable(tmp / 'pairs_0.npz')
assert (edited.walls != plain.walls).sum() == 4 and json.loads(str(np.load(tmp / 'pairs_100.npz')['source']))['edit'] == edits[100]
z100 = np.load(tmp / 'trajectories_100.npz')
assert all(tuple(c) in edited.idx for c in z100['cells'][z100['cells'][..., 0] >= 0])
try:
    PairCache(tmp / 'pairs_100.npz', env_cfg, 100)                              # without its edits: another layout
    raise AssertionError('should fail')
except ValueError:
    pass

In [ ]:
# variations: the counts drawn from each seed, the same in every env made from the config; none: the constructor's
cfg15 = {'_target_': 'stable_marl.envs.multigrid.FindGoalEnv', 'agents': 2, 'size': 15, 'num_obstacles': 6, 'n_clutter': 10}
counts = lambda cfg, seed: (lambda e: (e.reset(seed=seed), (int(e.unwrapped._variation('obstacles.number')),
                                                           int(e.unwrapped._variation('clutter.number'))))[1])(make_env(cfg))
walls_of = lambda cfg, seed: (lambda e: (e.reset(seed=seed), tuple(free_cells(e)))[1])(make_env(cfg))
varied = {**cfg15, 'variations': ['obstacles.number', 'clutter.number']}
assert all(counts(cfg15, s) == (6, 10) for s in range(5))
drawn = [counts(varied, s) for s in range(12)]
assert drawn == [counts(varied, s) for s in range(12)] and len(set(drawn)) > 6          # reproducible, and varied
assert all(0 <= o <= 12 and 0 <= c <= 20 for o, c in drawn)
assert walls_of(varied, 3) == walls_of(varied, 3) != walls_of(cfg15, 3)
rng = np.random.default_rng(0)                                                        # reset_in_layout keeps the walls
env = make_env(varied)
for _ in range(5):
    reset_in_layout(env, 3, rng)
    u = env.unwrapped
    assert tuple(sorted(free_cells(env))) == tuple(sorted(walls_of(varied, 3)))
    assert all(abs(a.state.pos[0] - u.goal_pos[0]) + abs(a.state.pos[1] - u.goal_pos[1]) >= 5 for a in u.agents)
try:
    make_env({**cfg15, 'variations': ['obstacle.number']})
    raise AssertionError('an unknown variation should fail')
except ValueError as e:
    assert 'unknown variations' in str(e)

The cache in an experiment: a pair never visited is traced when first needed, and kept.

In [ ]:
from comm_core import Link, Message, Transmission

In [ ]:
cache = PairCache(tmp / 'pairs_0.npz', env_cfg, 0, tracer=stand_in(table.walls))
fading = CachedRayTracedFading(cache)
new = next((a, b) for a in cache.cells for b in cache.cells if a != b and _pair(a, b) not in cache.H)
fading.set_positions({'agent_0': new[0], 'agent_1': new[1]})
transmission = Transmission(Message('agent_0', 'agent_1', payload=0), 'agent_0', 'agent_1', start_time=0)
before = len(cache.H)
h = fading.coefficient(transmission, Link('agent_0', 'agent_1'))
assert len(cache.H) == before + 1 and np.isclose(abs(h) ** 2, cache.gain(*new))
fading.coefficient(transmission, Link('agent_0', 'agent_1'))                   # now cached: nothing traced
assert len(cache.H) == before + 1 and cache.missing([new]) == []
fading.save()
assert _pair(*new) in ChannelTable(tmp / 'pairs_0.npz').pairs